# Downstream Analysis Example (SCLC)

This notebook demonstrates a lightweight downstream analysis workflow using the **merged cohort CSV** produced by the toolkit.

It covers:
- Loading the merged dataset
- Basic cohort summary (counts, age/date ranges)
- Therapy regimen distribution (if present)
- Mutation flag frequencies (if present)
- Expression signature exploration (if present)
- A simple PCA on expression/signature features

> If your run output file has a different name/path, update `DATA_PATH` below.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_PATH = "../dsl_sclc_dataset.csv"  # update if needed
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
# Basic cohort summary
print("Rows:", len(df))
print(
    "Unique patients:",
    df.get("patient_id", pd.Series(dtype=str)).nunique()
    if "patient_id" in df.columns
    else "(missing)",
)
print(
    "Unique samples:",
    df.get("sample_id", pd.Series(dtype=str)).nunique()
    if "sample_id" in df.columns
    else "(missing)",
)

if "age_at_collection" in df.columns:
    print(
        "Age (min/median/max):",
        df["age_at_collection"].min(),
        df["age_at_collection"].median(),
        df["age_at_collection"].max(),
    )

if "collection_date" in df.columns:
    dates = pd.to_datetime(df["collection_date"], errors="coerce")
    print("Collection date range:", dates.min(), "→", dates.max())

In [ ]:
# Therapy distribution (only if your SQL join included regimen)
if "regimen" in df.columns:
    counts = df["regimen"].fillna("(missing)").value_counts().head(15)
    counts.plot(kind="bar")
    plt.title("Top therapy regimens (top 15)")
    plt.ylabel("# samples")
    plt.tight_layout()
    plt.show()
else:
    print(
        "No 'regimen' column found. Run with --therapy-mode join_table if you have therapy table support."
    )

In [ ]:
# Mutation flag frequencies (if present)
mut_cols = [c for c in df.columns if c.startswith("mut_")]
if mut_cols:
    freq = df[mut_cols].fillna(0).mean().sort_values(ascending=False)
    freq.plot(kind="bar")
    plt.title("Mutation flag frequency")
    plt.ylabel("Fraction of samples")
    plt.tight_layout()
    plt.show()
else:
    print(
        "No mutation flag columns found. Run with --include-variants and ensure your variants TileDB has a 'GENE' annotation."
    )

In [ ]:
# Signature exploration (if present)
sig_cols = [c for c in df.columns if c.startswith("SCLC_")]
if sig_cols:
    df[sig_cols].describe().T
else:
    print("No signature columns found. Run with --compute-signatures.")

In [ ]:
# Pairwise signature scatter (first two signatures)
if len(sig_cols) >= 2:
    x, y = sig_cols[0], sig_cols[1]
    plt.scatter(df[x], df[y], alpha=0.6)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.title(f"Signature scatter: {x} vs {y}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Simple PCA on numeric features (expression genes + signatures + mutation flags)
num_cols = []
num_cols += [c for c in df.columns if c in sig_cols]
num_cols += [c for c in df.columns if c.startswith("mut_")]

# Also include individual gene expression columns if present (heuristic: all-caps gene symbols)
for c in df.columns:
    if c.isupper() and c not in ("GT", "QUAL") and c not in ("ID",):
        # avoid obvious non-expression
        if c not in ("CHROM", "POS", "REF", "ALT", "GENE"):
            num_cols.append(c)

num_cols = sorted(set(num_cols))
print("Using numeric columns:", num_cols)

if num_cols:
    X = df[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy()
    # center
    X = X - X.mean(axis=0, keepdims=True)
    # SVD PCA
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    pcs = U[:, :2] * S[:2]
    plt.scatter(pcs[:, 0], pcs[:, 1], alpha=0.6)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title("PCA of expression/signature/mutation features")
    plt.tight_layout()
    plt.show()
else:
    print(
        "No numeric feature columns detected. Ensure you ran with --compute-signatures and/or your expression pivot produced gene columns."
    )